# SFT benchmark — `sft_bench.py`

Generative chat benchmark for the **SFT** model: asks the 49 questions in
`sparky/bench/sft_bench.jsonl` using the exact trained chat template, reads the
generated answers, and grades them. Runs **three v2 checkpoints** back to back
(`sft_best` = best val @ step 1200, `sft_epoch1`, `sft_epoch2`) and prints a
side-by-side per-category table at the end.

- **System prompt is empty** — that is the trained distribution (every prose
  source has no system prompt). The old "You are a helpful assistant…" prompt
  appeared in zero training rows.
- **Deterministic graders** (numeric / keywords / regex / contains / refusal) always run.
- **LLM judge** (DeepSeek v4-flash) grades the 9 open-ended items if `DEEPSEEK_API_KEY`
  is in Colab Secrets — set it, otherwise creative/summarization stay ungraded.

Needs a **GPU** runtime. Loads checkpoints straight from Drive.

In [ ]:
# Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
CKPTS = {   # name -> checkpoint; each gets its own results subfolder
    'sft_best':   f'{SYNAPSE_DIR}/sft_checkpoints/v3_15source/sft_best.pth',    # best overall val (step 1200)
    'sft_epoch1': f'{SYNAPSE_DIR}/sft_checkpoints/v2_12source/sft_best.pth',   # v2 best (comparison)
    'sft_epoch2': f'{SYNAPSE_DIR}/sft_checkpoints/v3_15source/sft_epoch1.pth',  # v3 endpoint
}
TOKENIZER     = f'{SYNAPSE_DIR}/tokenizer_out/tokenizer.json'
PRETRAIN_CKPT = f'{SYNAPSE_DIR}/checkpoints/synapse_2b_d2560_l28.pth'  # for optional compare
RESULTS_DIR   = f'{SYNAPSE_DIR}/sft_bench_results'                # results persisted to Drive
for k, v in CKPTS.items(): print(f'{k:11s} {v}')
print('TOKENIZER  ', TOKENIZER)

In [ ]:
# Clone/pull repo (brings sft_bench.py, sparky_model.py, chat template, bench file)
import subprocess, sys
REPO_DIR = '/content/synapse_repo'
REPO_URL = 'https://github.com/ajencinas/synapse.git'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('repo exists — pulling latest'); subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
else:
    print('cloning'); subprocess.run(['git','clone','--depth=1',REPO_URL,REPO_DIR],check=True)
assert os.path.isfile(os.path.join(REPO_DIR,'sparky','sft_bench.py')), 'sft_bench.py missing — check branch'
print('REPO_DIR =', REPO_DIR)

In [ ]:
# Deps + GPU check
!pip install -q tokenizers openai
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime → GPU')

In [ ]:
# Preflight: confirm every checkpoint + tokenizer exist on Drive
ok = True
for label, p in list(CKPTS.items()) + [('tokenizer', TOKENIZER)]:
    present = os.path.exists(p); ok = ok and present
    print(('  OK ' if present else 'MISS ') + f'{label}: {p}')
assert ok, 'missing inputs above'

In [ ]:
# OPTIONAL: LLM judge (DeepSeek V4 by default). Reads the key from Colab Secrets
# (🔑 left sidebar → add DEEPSEEK_API_KEY, enable notebook access). Blank = the
# 40/49 deterministic prompts still grade with no key.
import os
JUDGE_PROVIDER = 'deepseek'   # 'deepseek' (deepseek-v4-flash) or 'openai' (gpt-4o-mini)
KEY_NAME = 'DEEPSEEK_API_KEY' if JUDGE_PROVIDER=='deepseek' else 'OPENAI_API_KEY'
try:
    from google.colab import userdata
    if userdata.get(KEY_NAME): os.environ[KEY_NAME] = userdata.get(KEY_NAME)
except Exception:
    pass
# os.environ[KEY_NAME] = '...'   # or paste here instead of using Secrets
if os.environ.get(KEY_NAME):
    print(f'LLM judge ENABLED ({JUDGE_PROVIDER})')
else:
    print('LLM judge disabled — deterministic graders only (40/49 prompts)')

In [ ]:
# Run the bench on each checkpoint. Greedy decode (top-k=1) for reproducible grading.
# System prompt: EMPTY (the trained distribution) — sft_bench's default since v2.
JUDGE    = '' if os.environ.get(KEY_NAME) else '--no-judge'
PROVIDER = f'--judge-provider {JUDGE_PROVIDER}'
COMPARE  = ''   # e.g. f'--compare-pretrain "{PRETRAIN_CKPT}"' to bench the base model too (once is enough)
for name, ckpt in CKPTS.items():
    out_dir = f'{RESULTS_DIR}/{name}'
    cmd = (f'cd {REPO_DIR}/sparky && python sft_bench.py '
           f'--ckpt "{ckpt}" --tokenizer "{TOKENIZER}" --system "" '
           f'--output-dir "{out_dir}" {PROVIDER} {JUDGE} {COMPARE}')
    print('\n' + '#' * 70 + f'\n# {name}\n' + '#' * 70 + '\n' + cmd)
    !{cmd}
    COMPARE = ''   # only compare the base model on the first checkpoint

In [ ]:
# Side-by-side: newest results file per checkpoint -> per-category pass/total
import glob, json, collections
table = {}; overall = {}
for name in CKPTS:
    files = sorted(glob.glob(f'{RESULTS_DIR}/{name}/*.json'), key=os.path.getmtime)
    if not files: print('no results for', name); continue
    rows = json.load(open(files[-1]))['results']['SFT']
    per = collections.defaultdict(lambda: [0, 0])
    for r in rows:
        if r['verdict'] is None: continue
        per[r['category']][1] += 1; per[r['category']][0] += int(bool(r['verdict']))
    table[name] = per
    overall[name] = (sum(v[0] for v in per.values()), sum(v[1] for v in per.values()))
cats = sorted({c for per in table.values() for c in per})
names = list(table)
print(f"{'category':24s}" + ''.join(f'{n:>12s}' for n in names))
for c in cats:
    print(f'{c:24s}' + ''.join(f"{table[n][c][0]}/{table[n][c][1]:<3d}".rjust(12) if c in table[n] else ' ' * 12 for n in names))
print(f"{'OVERALL':24s}" + ''.join(f'{overall[n][0]}/{overall[n][1]} ({100*overall[n][0]//max(1,overall[n][1])}%)'.rjust(12) for n in names))
print('\nfailures per checkpoint:')
for n in names:
    rows = json.load(open(sorted(glob.glob(f'{RESULTS_DIR}/{n}/*.json'), key=os.path.getmtime)[-1]))['results']['SFT']
    print(f'  {n:11s}', [r['id'] for r in rows if r['verdict'] is False])

## Notes
- **Which checkpoints**: edit `CKPTS` in cell 1. `sft_latest.pth` has the same
  weights as `sft_epoch2.pth` (plus optimizer state) — no need to bench both.
- **Base-vs-SFT**: set `COMPARE = f'--compare-pretrain "{PRETRAIN_CKPT}"'` to run
  the pretrain model on the same prompts (only on the first checkpoint).
- **Judge**: deterministic graders cover numeric/keyword/regex items offline; the
  DeepSeek judge adds rubric-scored grades for the 9 open-ended items.
- Base-model leaderboard numbers are a different tool: `sparky_leaderboard_colab.ipynb`.